# SatDetection — Fast Colab Training Notebook

**Multi-task U-Net + ResNet-50 for building and road segmentation**

This version is rewritten for Google Colab T4 GPU speed:
- Enables CUDA/cuDNN speed settings
- Copies dataset tiles from Google Drive to `/content`
- Trains from local Colab storage instead of Drive
- Uses a larger batch size by default
- Keeps checkpoints and outputs saved in Google Drive

Before running: **Runtime → Change runtime type → T4 GPU**


## 1. Check GPU and Enable Speed Settings


In [ ]:
import torch

print(f"GPU available: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"Device : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: GPU is not available. Go to Runtime > Change runtime type > T4 GPU.")

# Speed settings for CNN/U-Net style models
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("Using device:", device)
print("CUDA/cuDNN speed settings enabled.")


## 2. Install Dependencies


In [ ]:
%%capture
!pip install segmentation-models-pytorch albumentations rasterio wandb pyyaml tqdm


## 3. Mount Google Drive and Set Paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import glob
import shutil
import numpy as np

BASE_DIR       = '/content/drive/MyDrive/SatDetection'
DATA_DIR       = os.path.join(BASE_DIR, 'data')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
VISUALS_DIR    = os.path.join(BASE_DIR, 'outputs/visuals')

DRIVE_RAW_DIR   = os.path.join(DATA_DIR, 'landcoverai/raw')
DRIVE_TILES_DIR = os.path.join(DATA_DIR, 'landcoverai/tiles')

# Local Colab paths. These are faster than Google Drive.
LOCAL_TILES_DIR = '/content/landcoverai_tiles'

for d in [
    DATA_DIR,
    CHECKPOINT_DIR,
    VISUALS_DIR,
    DRIVE_RAW_DIR,
    os.path.join(DRIVE_TILES_DIR, 'images'),
    os.path.join(DRIVE_TILES_DIR, 'masks'),
]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'Base directory       : {BASE_DIR}')
print(f'Drive raw data       : {DRIVE_RAW_DIR}')
print(f'Drive tiles          : {DRIVE_TILES_DIR}')
print(f'Local training tiles : {LOCAL_TILES_DIR}')


## 4. Clone Project Repository


In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/SatDetection.git'  # <-- change this to your real GitHub repo URL

if REPO_URL == 'https://github.com/YOUR_USERNAME/SatDetection.git':
    print("WARNING: Update REPO_URL before running this cell.")
else:
    if not os.path.exists('/content/SatDetection'):
        !git clone $REPO_URL /content/SatDetection
    else:
        !git -C /content/SatDetection pull

    sys.path.insert(0, '/content/SatDetection/src')
    print('Repo ready.')


## 5. W&B Login


In [ ]:
import wandb
wandb.login()


## 6. Inspect Raw LandCover.ai Files

Upload/extract LandCover.ai to:

```text
MyDrive/SatDetection/data/landcoverai/raw/
```

Expected format:

```text
raw/
  M-33-20-C-c-3-4.tif
  M-33-20-C-c-3-4_m.tif
```


In [ ]:
all_tifs    = sorted(glob.glob(os.path.join(DRIVE_RAW_DIR, '**/*.tif'), recursive=True))
image_tifs  = [f for f in all_tifs if not f.endswith('_m.tif')]
mask_tifs   = [f for f in all_tifs if f.endswith('_m.tif')]

print(f'Raw folder  : {DRIVE_RAW_DIR}')
print(f'Total .tifs : {len(all_tifs)}')
print(f'Images      : {len(image_tifs)}')
print(f'Masks       : {len(mask_tifs)}')

if image_tifs:
    print(f'Example image: {os.path.basename(image_tifs[0])}')
if mask_tifs:
    print(f'Example mask : {os.path.basename(mask_tifs[0])}')


## 7. Tile the Dataset into `.npy` Files


In [ ]:
# Run tiling only if tiles do not already exist in Drive.
# This saves the prepared tiles in Drive so you do not have to tile again every runtime.

from utils import tile_all

existing_images = len(glob.glob(os.path.join(DRIVE_TILES_DIR, 'images/*.npy')))
existing_masks  = len(glob.glob(os.path.join(DRIVE_TILES_DIR, 'masks/*.npy')))

if existing_images > 0 and existing_images == existing_masks:
    print(f'Tiles already exist in Drive: {existing_images}. Skipping tiling.')
else:
    assert len(image_tifs) > 0, 'No images found in raw folder. Upload LandCover.ai data first.'
    assert len(mask_tifs) > 0, 'No masks found in raw folder. Upload LandCover.ai masks first.'

    print('Tiling raw LandCover.ai files...')
    total = tile_all(DRIVE_RAW_DIR, DRIVE_RAW_DIR, DRIVE_TILES_DIR, tile_size=256, stride=256)
    print(f'Done. Total tiles: {total}')


## 8. Verify Drive Tiles


In [ ]:
drive_image_tiles = sorted(glob.glob(os.path.join(DRIVE_TILES_DIR, 'images/*.npy')))
drive_mask_tiles  = sorted(glob.glob(os.path.join(DRIVE_TILES_DIR, 'masks/*.npy')))

n_images = len(drive_image_tiles)
n_masks  = len(drive_mask_tiles)

print(f'Drive image tiles : {n_images}')
print(f'Drive mask tiles  : {n_masks}')

assert n_images == n_masks, 'Mismatch between image tiles and mask tiles.'
assert n_images > 0, 'No tiles found. Check raw folder and re-run tiling.'

sample_img  = np.load(drive_image_tiles[0])
sample_mask = np.load(drive_mask_tiles[0])

print(f'Image shape : {sample_img.shape}  dtype: {sample_img.dtype}')
print(f'Mask shape  : {sample_mask.shape}  dtype: {sample_mask.dtype}')
print(f'Mask values : building max={sample_mask[0].max():.0f}  road max={sample_mask[1].max():.0f}')


## 9. Copy Tiles from Google Drive to Local Colab Storage


In [ ]:
# Training directly from Google Drive can be slow.
# This copies the tile folder to /content, which is much faster.

if os.path.exists(LOCAL_TILES_DIR):
    local_images = glob.glob(os.path.join(LOCAL_TILES_DIR, 'images/*.npy'))
    local_masks  = glob.glob(os.path.join(LOCAL_TILES_DIR, 'masks/*.npy'))

    if len(local_images) == n_images and len(local_masks) == n_masks:
        print(f'Local tiles already exist: {len(local_images)} images, {len(local_masks)} masks.')
    else:
        print('Local tiles are incomplete. Recopying...')
        shutil.rmtree(LOCAL_TILES_DIR)
        shutil.copytree(DRIVE_TILES_DIR, LOCAL_TILES_DIR)
else:
    print('Copying tiles from Drive to local Colab storage...')
    shutil.copytree(DRIVE_TILES_DIR, LOCAL_TILES_DIR)

local_images = glob.glob(os.path.join(LOCAL_TILES_DIR, 'images/*.npy'))
local_masks  = glob.glob(os.path.join(LOCAL_TILES_DIR, 'masks/*.npy'))

print(f'Local image tiles : {len(local_images)}')
print(f'Local mask tiles  : {len(local_masks)}')
print(f'Local tiles path  : {LOCAL_TILES_DIR}')

assert len(local_images) == len(local_masks), 'Local image/mask tile mismatch.'
assert len(local_images) > 0, 'No local tiles found.'


## 10. Training Config


In [ ]:
BASE_CONFIG = {
    # Use local Colab storage for faster training
    'landcoverai_root':        LOCAL_TILES_DIR,

    # Save important outputs back to Drive
    'checkpoint_dir':          CHECKPOINT_DIR,
    'visuals_dir':             VISUALS_DIR,

    'wandb_project':           'SatDetection',
    'epochs':                  50,

    # T4 usually handles 32 for 256x256 tiles, but reduce to 16 if CUDA runs out of memory
    'batch_size':              32,

    # Colab works well with 2. If it crashes or hangs, use 0.
    'num_workers':             2,

    'early_stopping_patience': 8,
    'w_building':              1.0,
    'w_road':                  2.0,
}

EXPERIMENTS = [
    {'pretrained': False, 'name': 'Model A — Random Init',        'lr': 0.00005},
    {'pretrained': True,  'name': 'Model B — ImageNet Pretrained', 'lr': 0.0001},
]

print('Experiment plan:')
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f'  Run {i}: {exp["name"]} | lr={exp["lr"]}')

print('\nTraining tiles will be loaded from:')
print(BASE_CONFIG['landcoverai_root'])


## 11. Final Diagnostic Before Training


In [ ]:
tiles_path = os.path.join(BASE_CONFIG['landcoverai_root'], 'images')
found = sorted(glob.glob(os.path.join(tiles_path, '*.npy')))

print(f'Tiles path : {tiles_path}')
print(f'Tiles found: {len(found)}')
print(f'Batch size : {BASE_CONFIG["batch_size"]}')
print(f'Workers    : {BASE_CONFIG["num_workers"]}')
print(f'Device     : {device}')

if len(found) == 0:
    raise RuntimeError('No local tiles found. Re-run the copy-to-local-storage cell.')
else:
    print('Ready to train from local Colab storage.')


## 12. Training Function


In [ ]:
from train import train

def run_experiment(exp_index):
    exp = EXPERIMENTS[exp_index - 1]

    config = {
        **BASE_CONFIG,
        'pretrained': exp['pretrained'],
        'lr': exp['lr'],
    }

    print(f"\n{'='*60}")
    print(f"Run {exp_index}: {exp['name']} | lr={exp['lr']}")
    print(f"{'='*60}")

    checkpoint = train(config)

    print(f"Checkpoint: {checkpoint}")
    return checkpoint


## 13. Run Model A — Random Initialization


In [ ]:
ckpt_1 = run_experiment(1)


## 14. Run Model B — ImageNet Pretrained


In [ ]:
ckpt_2 = run_experiment(2)


## 15. Evaluation


In [ ]:
from evaluate import evaluate

def eval_experiment(exp_index, checkpoint_path):
    exp = EXPERIMENTS[exp_index - 1]

    config = {
        **BASE_CONFIG,
        'pretrained': exp['pretrained'],
    }

    print(f"\n{'='*60}")
    print(f"Evaluating Run {exp_index}: {exp['name']}")
    print(f"{'='*60}")

    return evaluate(config, checkpoint_path, save_visuals=True, n_visuals=5)


## 16. Evaluate Model A


In [ ]:
results_1 = eval_experiment(1, ckpt_1)


## 17. Evaluate Model B


In [ ]:
results_2 = eval_experiment(2, ckpt_2)


## 18. Results Summary


In [ ]:
import pandas as pd

rows = []
for exp, results in [
    (EXPERIMENTS[0], results_1),
    (EXPERIMENTS[1], results_2),
]:
    rows.append({
        'Model':        exp['name'],
        'IoU Building': f"{results['iou_building']:.4f}",
        'IoU Road':     f"{results['iou_road']:.4f}",
        'Mean IoU':     f"{(results['iou_building'] + results['iou_road']) / 2:.4f}",
        'F1 Building':  f"{results['f1_building']:.4f}",
        'F1 Road':      f"{results['f1_road']:.4f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df


## 19. If You Get CUDA Out of Memory


In [ ]:
# Run this cell only if training crashes with CUDA out of memory.
# Then re-run the training cell.

BASE_CONFIG['batch_size'] = 16
print("Batch size reduced to:", BASE_CONFIG['batch_size'])
